<a href="https://colab.research.google.com/github/priyanshu-kr/LLM_projects/blob/main/MyPromptEngineeringLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Prompt Engineering Lab

Goal: Learn how prompts affect an LLM's dialogue summaries.

In [1]:
!pip install -q datasets transformers

In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/dialogsum")

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})


In [4]:
example = dataset["test"][0]

print("DIALOGUE:")
print(example["dialogue"])

print("\nHUMAN SUMMARY:")
print(example["summary"])

DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with th

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Baseline

We are giving the model only the dialogue, not the human summary and not an instruction such as "summarize this". This is our baseline.

In [6]:
dialogue = dataset["test"][0]["dialogue"]

inputs = tokenizer(dialogue, return_tensors = "pt", truncation = True)

outputs = model.generate(**inputs, max_new_tokens=50)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)

print(generated_text)

#Person1: Ms. Dawson, please take dictation for me.


In [7]:
print(dialogue)

#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with the memo. Wh

Now, we will give the model an explicit instruction.

In [13]:
prompt = f"""Here is a dialogue:
{dialogue}

Write a short summary!"""

inputs = tokenizer(prompt, return_tensors = 'pt', truncation = True)

outputs = model.generate(**inputs, max_new_tokens = 180)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)

print("Generated Text:")
print(repr(generated_text))

Generated Text:
"Person1 wants to send an intra-office memo to all employees. It's about a new policy on communications. Employees who use Instant Messaging during working hours will be warned and placed on probation. Employees who continue to use Instant Messaging will face termination."


In [15]:
prompt = f"""Summarize the following dialogue in two sentences.
State whether the communication policy applies to internal communication, external communication, or both. Also include the consequence for violating it.

Dialogue:
{dialogue}

Summary:"""

inputs = tokenizer(prompt, return_tensors = 'pt', truncation = True)

outputs = model.generate(**inputs, max_new_tokens = 150)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)

print("Generated Text:")
print(repr(generated_text))

Generated Text:
'#Person1 wants to restrict the use of Instant Messaging in the office. #Person2 is to type up and distribute the memo to all employees by this afternoon.'
